# Train a custom "hey gideon" wake word model

**Status**: a `hey_gideon` model is already trained and committed at
`../models/hey_gideon.onnx` (trained locally on CPU, not via this notebook -
see `../plan.md` and `../training/README.md`'s "Local CPU training" section
for what that took and the measured results). This notebook is kept as the
documented route to **retrain** - e.g. with a larger `n_samples` if the
current model's false-positive rate is a problem in real use - either here
in Colab or adapted for local use.

This notebook is adapted from openWakeWord's official [`automatic_model_training.ipynb`](https://github.com/dscripka/openWakeWord/blob/main/notebooks/automatic_model_training.ipynb), parameterized for this repo's target phrase (**"hey gideon"**) instead of the upstream example phrase. It's been patched throughout for several breakages discovered in openWakeWord's and its dependencies' current state (see each "Compatibility patch"/"Compatibility fix" cell for details) - none of these patches have been run end-to-end in an actual Colab environment, only reasoned through and adapted from what fixed the same issues locally, so treat a first Colab run of this notebook as still somewhat unverified.

**Run this in Google Colab with a GPU runtime** (Runtime -> Change runtime type -> T4 GPU). It downloads several GB of datasets and trains for a while, so a local CPU-only machine is not a good fit.

**Important**: automatic training currently only works on Linux (Piper TTS requirement) - Colab's backend is Linux, so this is fine there.

At the end you'll have `my_custom_model/hey_gideon.onnx` - download that file and drop it into `modules/02-wake-word/models/hey_gideon.onnx` in this repo, then point `config/config.yaml`'s `wake_word.model` at that path. See `training/README.md` alongside this notebook for the full hand-off steps.

# Environment Setup

In [ ]:
# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator

# Pin to the last commit before piper-sample-generator's "Move to package"
# restructuring (2026-03-12, commit 1a8c49b), which deleted the root-level
# generate_samples.py that openwakeword/train.py imports
# (`from generate_samples import generate_samples`). Cloning the default
# branch as-is breaks immediately with `ModuleNotFoundError: No module
# named 'generate_samples'`.
!cd piper-sample-generator && git checkout c9d824c0e2cce8bdeb000c219dc9cbc84df086ea

!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad
# At the pinned commit, generate_samples.py unconditionally imports
# `from piper import PiperVoice, SynthesisConfig` (only actually used by a
# sibling ONNX-based function, not by generate_samples() itself, but still
# required at import time) - install the `piper-tts` package to satisfy it.
!pip install piper-tts

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword

# openwakeword's setup.py unconditionally requires speexdsp-ns, but that
# package only ships prebuilt wheels (on GitHub releases, not even PyPI) for
# old CPython versions - it has none for the Python version current Colab
# runtimes ship, so a normal `pip install -e .` fails outright with
# "No matching distribution found for speexdsp-ns". speexdsp-ns is only used
# for an opt-in noise-suppression feature (openwakeword/model.py, behind
# `enable_speex_noise_suppression`, off by default) that train.py never
# touches, so it's safe to install without deps and pull in the rest of the
# real dependencies manually instead.
!{sys.executable} -m pip install -e ./openwakeword --no-deps
!{sys.executable} -m pip install "onnxruntime>=1.10.0,<2" "ai-edge-litert>=2.0.2,<3" "tqdm>=4.0,<5.0" "scipy>=1.3,<2" "scikit-learn>=1,<2" "requests>=2.0,<3"

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
# datasets==2.14.6 (the upstream notebook's original pin) fails to import
# under current pyarrow ("AttributeError: module 'pyarrow' has no attribute
# 'PyExtensionType'" - a pyarrow API `datasets` 2.14.x still used has since
# been removed). The current latest `datasets` (5.x) fixes that but then
# requires the `torchcodec` package for audio decoding, which itself needs
# system-level ffmpeg shared libraries (libavutil.so.56/57) that aren't
# guaranteed present - avoid that whole chain by pinning to 2.19.0, which
# predates the torchcodec requirement and postdates the pyarrow removal.
!pip install "datasets==2.19.0"
!pip install deep-phonemizer==0.0.19
# `pronouncing` imports `pkg_resources` (from setuptools), which recent
# setuptools releases (81+) have dropped entirely - pin an older setuptools
# that still ships it.
!pip install "setuptools<81"
# PyTorch's newer ONNX exporter (used by train.py's final export step)
# needs this as a separate package.
!pip install onnxscript

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models")
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

In [ ]:
# Compatibility patch: torch_audiomentations==0.11.0 (pinned above) calls
# torchaudio.set_audio_backend(...), an API removed in the newer torchaudio
# that ships with current Colab runtimes (backend selection is automatic
# now). Locate the file via find_spec (NOT a plain `import
# torch_audiomentations`, which would immediately hit the same crash) and
# neutralize the one offending line - it's a no-op in modern torchaudio.
import importlib.util
import pathlib

spec = importlib.util.find_spec("torch_audiomentations")
p = pathlib.Path(spec.origin).parent / "utils" / "io.py"
src = p.read_text()
old = 'torchaudio.set_audio_backend("soundfile")'
if old in src:
    src = src.replace(old, "pass  # patched: set_audio_backend removed in modern torchaudio")
    p.write_text(src)
    print("patched:", p)
else:
    print("pattern not found (already patched, or upstream changed) - check manually:", p)

In [ ]:
# Compatibility patch: at the pinned piper-sample-generator commit,
# generate_samples()'s `model` parameter has no default (a default pointing
# at piper-sample-generator/models/en_US-libritts_r-medium.pt existed in
# older versions but was removed in the same commit that upgraded to
# torch 2). openwakeword/train.py's four calls to generate_samples() were
# written against the old default-having version and never pass `model=`,
# so they'd fail with `TypeError: generate_samples() missing 1 required
# positional argument: 'model'`. Patch train.py to compute the model path
# once and pass it explicitly at each call site.
import pathlib

train_py = pathlib.Path("openwakeword/openwakeword/train.py")
src = train_py.read_text()

anchor = "from generate_samples import generate_samples"
assert anchor in src, "train.py's import of generate_samples has changed - re-check this patch"
if "piper_model_path = " not in src:
    src = src.replace(
        anchor,
        anchor + '\n\n    piper_model_path = os.path.join(config["piper_sample_generator_path"], "models", "en_US-libritts_r-medium.pt")',
        1,
    )

old_call = "generate_samples("
new_call = "generate_samples(model=piper_model_path, "
n = src.count(old_call) - src.count(new_call)
if n > 0:
    src = src.replace(old_call, new_call)
    print(f"patched {n} generate_samples() call site(s)")
else:
    print("generate_samples() calls already patched")

# Second, unrelated bug in this same file: --convert_to_tflite uses
# action="store_true" with default="False" - a non-empty STRING, which is
# truthy in Python, so `if args.convert_to_tflite:` always runs regardless
# of whether the flag is passed. That crashes the whole training run with
# ModuleNotFoundError: onnx_tf right after the (successful) ONNX export
# finishes, unless tensorflow/onnx_tf happen to be installed - this repo
# only ever loads the .onnx file, so we don't want that dependency. Patch
# the default to a real bool so the conversion is actually optional.
old_default = '''    parser.add_argument(
        "--convert_to_tflite",
        help="Convert the trained ONNX model to TFLite format",
        action="store_true",
        default="False",
        required=False
    )'''
if old_default in src:
    src = src.replace(old_default, old_default.replace('default="False"', "default=False"))
    print('patched --convert_to_tflite default (string "False" -> real bool False)')
else:
    print("--convert_to_tflite default already patched (or upstream changed) - check manually")

train_py.write_text(src)

In [ ]:
# Compatibility patch: current torchaudio routes ALL audio I/O
# (torchaudio.load/.info, used throughout openwakeword/data.py) through
# torchcodec, which needs system ffmpeg shared libraries
# (libavdevice/libavutil at specific versions) that may not be present on
# a given Colab runtime and aren't installable without sudo. Every audio
# file this pipeline touches is a plain WAV, so bypass torchcodec entirely
# by shimming both functions to use soundfile instead. Written as a
# sitecustomize.py (auto-imported by every Python process using this
# site-packages, including the separate `!{sys.executable} train.py ...`
# subprocess invocations below) rather than an in-notebook monkeypatch,
# which would only affect this notebook's own kernel process.
import site

site_packages = site.getsitepackages()[0]
with open(f"{site_packages}/sitecustomize.py", "w") as f:
    f.write('''
import soundfile as sf
import torch
import torchaudio


class _AudioMetaDataShim:
    def __init__(self, info):
        self.sample_rate = info.samplerate
        self.num_frames = info.frames
        self.num_channels = info.channels
        self.bits_per_sample = 0
        self.encoding = "PCM_S"


def _info_via_soundfile(filepath, *args, **kwargs):
    try:
        info = sf.info(filepath)
    except Exception as e:
        raise RuntimeError(str(e))
    return _AudioMetaDataShim(info)


def _load_via_soundfile(filepath, *args, **kwargs):
    data, sr = sf.read(filepath, dtype="float32", always_2d=True)
    return torch.from_numpy(data.T).contiguous(), sr


torchaudio.load = _load_via_soundfile
torchaudio.info = _info_via_soundfile
''')
print("wrote sitecustomize.py shim to", site_packages)

In [ ]:
# Compatibility patch: deep-phonemizer (imported as `dp`) is used by
# generate_adversarial_texts() as a fallback for any word in --target-phrase
# not found in the standard pronunciation dictionary (an invented name, for
# instance - "hey gideon" itself doesn't need this, both words are
# in-dictionary, but a different phrase might). Its own checkpoint loader
# calls torch.load(...) without weights_only=False, which PyTorch 2.6+ now
# blocks by default with an UnpicklingError. Patch the one call site.
import importlib.util
import pathlib

spec = importlib.util.find_spec("dp.model.model")
p = pathlib.Path(spec.origin)
src = p.read_text()
old = "checkpoint = torch.load(checkpoint_path, map_location=device)"
new = "checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)"
if old in src:
    p.write_text(src.replace(old, new))
    print("patched deep-phonemizer:", p)
else:
    print("deep-phonemizer already patched (or upstream changed) - check manually:", p)

In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

# Download Data

Four types of data are needed: synthetic positive examples (generated below from the target phrase), synthetic adversarial examples, room-impulse-response/noise data for augmentation, and generic negative audio (via openWakeWord's precomputed feature set) plus a validation set for early stopping. All are fetched from HuggingFace.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
# Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
#
# The upstream notebook downloaded a raw `bal_train09.tar` shard directly -
# that file no longer exists (404): the `agkphysics/AudioSet` HF dataset was
# reorganized from raw per-shard .tar files into parquet format
# (data/bal_train/*.parquet) at some point after this notebook was written.
# Load it the same streaming way the MIT RIR cell above already does instead.
#
# For full-scale training, it's recommended to increase target_hours below
# (or download the full dataset) and even combine it with other background
# noise datasets (e.g., FSD50k, Freesound, etc.)

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

target_hours = 3  # AudioSet clips are ~10s each
target_n = target_hours * 3600 // 10
audioset_dataset = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for i, row in enumerate(tqdm(audioset_dataset, total=target_n)):
    name = row['video_id'] + ".wav"
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    if i + 1 >= target_n:
        break

# Free Music Archive dataset (https://github.com/mdeff/fma) - DROPPED.
#
# `rudraml/fma`'s dataset loading script is currently broken on both
# streaming (`ValueError: Cannot seek streaming HTTP file` - its custom
# loader script tries to seek into an HTTP-streamed zip to read metadata)
# and non-streaming access (attempting to materialize the dataset just
# times out) with current library versions. Rather than chase a fragile
# third-party dataset's breakage, background augmentation here relies on
# AudioSet + MIT RIRs only. If music-specific false positives show up during
# real testing, revisit adding a different music/noise dataset as a second
# `background_paths` entry.

# Define Training Configuration

Target phrase is **"hey gideon"**. Compared to openWakeWord's own toy example (1,000 samples / 10,000 steps, to keep the demo fast), this uses a larger sample count since this model is meant for real use, not just a demo - expect this to take noticeably longer on a free Colab GPU. `steps: 50000` and `target_false_positives_per_hour: 0.2` are already the config schema's own defaults (see `openwakeword/examples/custom_model.yml`), so they aren't overridden below. If Colab disconnects/times out, just re-run the training cells; `--generate_clips` and `--train_model` both resume from where they left off.

Note: `n_samples: 3000` below is still below openWakeWord's own recommendation in that same example file ("minimum of 20,000 recommended, often 100,000+ is best"). This is a reasonable starting point to get a working model without an extremely long training run, but if recall ends up weak (same problem as the current `hey_jarvis` pretrained model - see `../plan.md`), retraining with a larger `n_samples` is the first thing to try.

In [ ]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

In [ ]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hey gideon"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")  # -> "hey_gideon"
config["n_samples"] = 3000
config["n_samples_val"] = 2000
# steps and target_false_positives_per_hour are left at the schema's own
# defaults (50000 and 0.2) - not overridden here.

config["background_paths"] = ['./audioset_16k']  # fma dropped - see the Download Data cell above
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

Three steps, run in sequence. Each can simply be re-run if it's interrupted (e.g. by a Colab disconnect) - they pick up where they left off rather than starting over.

In [ ]:
# Step 1: Generate synthetic clips
# With n_samples=3000 this will take noticeably longer than the ~10 minute upstream
# example (which uses 1000) - plan for it to run a while on a free Colab T4.
# If generation fails or Colab disconnects, just run this command again - it
# continues generating until the number of files meets the targets in the config file.

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Compatibility fix: en_US-libritts_r-medium (the Piper voice used above)
# generates audio at 22050 Hz, but openWakeWord's whole augmentation/feature
# pipeline hardcodes 16000 Hz (e.g. augment_clips()'s `sr: int = 16000`
# default and its PitchShift's `sample_rate=16000`). Without this step,
# Step 2 below fails with "ValueError: Clip does not have the correct
# sample rate!" on the very first clip. Resample all 4 generated-clip
# directories to 16kHz in place, once, before augmenting.
import math
import soundfile as sf
from scipy.signal import resample_poly

TARGET_SR = 16000
for d in ["positive_train", "positive_test", "negative_train", "negative_test"]:
    d_path = os.path.join(config["output_dir"], config["model_name"], d)
    files = [f for f in os.listdir(d_path) if f.endswith(".wav")]
    n_resampled = 0
    for fname in files:
        fpath = os.path.join(d_path, fname)
        data, sr = sf.read(fpath, dtype="float32")
        if sr == TARGET_SR:
            continue
        g = math.gcd(TARGET_SR, sr)
        up, down = TARGET_SR // g, sr // g
        resampled = resample_poly(data, up, down)
        sf.write(fpath, resampled, TARGET_SR, subtype="PCM_16")
        n_resampled += 1
    print(f"{d}: resampled {n_resampled}/{len(files)} files")

In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Compatibility fix: PyTorch's newer ONNX exporter (used by train.py's
# final export step above) defaults to splitting large weight tensors into
# a companion `<model>.onnx.data` file, even for a model this tiny. Since
# this repo just wants one self-contained `.onnx` file, re-save with
# everything embedded inline.
import onnx

onnx_path = f"{config['output_dir']}/{config['model_name']}.onnx"
m = onnx.load(onnx_path, load_external_data=True)
onnx.save_model(m, onnx_path, save_as_external_data=False)
print("re-saved as a single inline file:", onnx_path)

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly.
# If so, run this cell to retry. Not required for this repo (detector.py forces the
# ONNX backend), but harmless to have both.

def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")

# Download the trained model

After training finishes, `my_custom_model/hey_gideon.onnx` (and `.tflite`) exist in the Colab filesystem. Download `hey_gideon.onnx` from the Colab file browser (folder icon in the left sidebar) and bring it back to this repo - see `training/README.md` for the exact hand-off steps (where to put the file and what to change in `config/config.yaml`).

In [ ]:
# Convenience cell: zip the onnx model so it's a single easy download
from google.colab import files
!cp my_custom_model/hey_gideon.onnx ./hey_gideon.onnx
files.download('hey_gideon.onnx')